### RAG pipelines- Data Ingestion to Vector DB piplines 

In [27]:
import os
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [28]:
from pathlib import Path
pdf_dir = Path("../data")
print(pdf_dir.resolve())        # shows absolute path
print(list(pdf_dir.rglob("*"))) # shows ALL files found

C:\Users\manas_978n0hd\OneDrive\Desktop\Full stack app development\VisaMentor-AI\data
[WindowsPath('../data/pdf_files'), WindowsPath('../data/Raw'), WindowsPath('../data/vector_store'), WindowsPath('../data/pdf_files/i-129instr.pdf'), WindowsPath('../data/pdf_files/i-765instr.pdf'), WindowsPath('../data/vector_store/0c262149-a2be-406d-9e37-ff4a7d8024ab'), WindowsPath('../data/vector_store/189905fd-4b52-4236-8845-ef05f6163661'), WindowsPath('../data/vector_store/1aef8839-050e-4a77-aaad-051592375e59'), WindowsPath('../data/vector_store/37785bca-3332-47f5-9f78-cac73d01d37b'), WindowsPath('../data/vector_store/3ef1f02e-28d9-4faa-ba89-e008eb4f64e0'), WindowsPath('../data/vector_store/499c116c-227d-4d3a-87b3-54ab44cea492'), WindowsPath('../data/vector_store/4e1a4772-c6a4-4648-ab26-18277dd58a98'), WindowsPath('../data/vector_store/989c464c-80a7-4518-8ac6-a7a36c1aef7a'), WindowsPath('../data/vector_store/bd3880f5-61c7-45b6-b613-8eb784f6d32a'), WindowsPath('../data/vector_store/bde881e4-7c75-49

In [29]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in directory"""

    all_documents =[]
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print (pdf_files)

    print(f" Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing : {pdf_file.name} ")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:   
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")

    return all_documents

all_pdf_documents = process_all_pdfs("../data")

[WindowsPath('../data/pdf_files/i-129instr.pdf'), WindowsPath('../data/pdf_files/i-765instr.pdf')]
 Found 2 PDF files to process
Processing : i-129instr.pdf 
  ✓ Loaded 32 pages
Processing : i-765instr.pdf 
  ✓ Loaded 25 pages


In [30]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.2 (Windows)', 'creationdate': '2026-02-19T08:18:51-05:00', 'author': 'USCIS', 'moddate': '2026-02-19T08:47:55-05:00', 'subject': 'Instructions for Petition for a Nonimmigrant Worker', 'title': 'Form I-129, Instructions for Petition for a Nonimmigrant Worker', 'trapped': '/False', 'source': '..\\data\\pdf_files\\i-129instr.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1', 'source_file': 'i-129instr.pdf', 'file_type': 'pdf'}, page_content='Instructions for Petition for Nonimmigrant Worker\nDepartment of Homeland Security\nU.S. Citizenship and Immigration Services\nForm I-129 Instructions   02/27/26 Page 1 of 32\nUSCIS\nForm I-129\nOMB No. 1615-0009\nExpires 12/31/2027\nTable of Contents  Page\nInstructions for Form I-129\nGeneral Information\nThe Purpose of Form I-129 ..................................................................................................................................

In [31]:
### Text splitting get into chunks

def split_documents(documents, chunk_size= 500, chunk_overlap=100):
    """Split Documents into smaller chunks for better RAG performance."""
    text_splitter= RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print("\n Example of Chunks")
        print(f"Content1: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs


In [32]:
chunks = split_documents(all_pdf_documents)

Split 57 documents into 542 chunks

 Example of Chunks
Content1: Instructions for Petition for Nonimmigrant Worker
Department of Homeland Security
U.S. Citizenship and Immigration Services
Form I-129 Instructions   02/27/26 Page 1 of 32
USCIS
Form I-129
OMB No. 161
Metadata: {'producer': 'Adobe PDF Library 18.0', 'creator': 'Adobe InDesign 21.2 (Windows)', 'creationdate': '2026-02-19T08:18:51-05:00', 'author': 'USCIS', 'moddate': '2026-02-19T08:47:55-05:00', 'subject': 'Instructions for Petition for a Nonimmigrant Worker', 'title': 'Form I-129, Instructions for Petition for a Nonimmigrant Worker', 'trapped': '/False', 'source': '..\\data\\pdf_files\\i-129instr.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1', 'source_file': 'i-129instr.pdf', 'file_type': 'pdf'}


### Embedding and vector store DB

In [33]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
class EmbeddingManager:
    """Handles document embedding generations using SentenceTransformer"""

    def __init__(self, model_name: str= "all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model"""
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape(len(texts),embedding_dimensions)
        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddinggs for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings

## Initialise the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager 


loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1772.07it/s]


Model loaded successfully. Embedding dimension: 384


### Vectore Store

In [35]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Delete existing collection to avoid duplicates on re-run
            try:
                self.client.delete_collection(name=self.collection_name)
                print("Cleared existing collection")
            except Exception:
                  pass

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    
    

Cleared existing collection
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [36]:
### Convert the text into embeddings

texts= [doc.page_content for doc in chunks]

## Generate Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Store in vector Database
vectorstore.add_documents(chunks,embeddings)


Generating embeddinggs for 542 texts...


Batches: 100%|██████████| 17/17 [00:12<00:00,  1.32it/s]


Generated embedding with shape: (542, 384)
Adding 542 documents to vector store...
Successfully added 542 documents to vector store
Total documents in collection: 542


## Retriever Pipeline from Vector Store

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generatscore_thresholding query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



SyntaxError: invalid syntax (2513886032.py, line 53)

In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("What is OPT?")

Retrieving documents for query: 'What is OPT?'
Top K: 5, Score threshold: 0.0
Generating embeddinggs for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 97.73it/s]

Generated embedding with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [ ]:
rag_retriever.retrieve("When I am eligible for OPT?")

Retrieving documents for query: 'When I am eligible for OPT?'
Top K: 5, Score threshold: 0.0
Generating embeddinggs for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 129.80it/s]

Generated embedding with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_83719eb3_344',
  'content': 'endorsed by the DSO before filing Form I-765.\nB. Post-Completion OPT--(c)(3)(B).  File Form I-765 up to 90 days before, but no later than 60 days after, your \nprogram end date.  Use Part 6. Additional Information to provide all previously used SEVIS numbers and \nevidence of any previously authorized CPT or OPT and the academic level at which it was authorized.',
  'metadata': {'moddate': '2025-08-26T07:45:17-04:00',
   'trapped': '/False',
   'subject': 'Instructions for Application for Employment Authorization',
   'file_type': 'pdf',
   'source': '..\\data\\pdf_files\\i-765instr.pdf',
   'content_length': 362,
   'creationdate': '2025-08-26T07:22:29-04:00',
   'page': 3,
   'source_file': 'i-765instr.pdf',
   'total_pages': 25,
   'doc_index': 344,
   'producer': 'Adobe PDF Library 17.0',
   'creator': 'Adobe InDesign 20.4 (Windows)',
   'page_label': '4',
   'author': 'FMB, USCIS',
   'title': 'Form I-765, Instructions for Application for

### RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM

        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.getenv("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")

        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context

        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length

        Returns:
            Generated response string
        """

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

            Context: {context}

            Question: {question}

            Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )

        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting

        Args:
            query: User question
            context: Retrieved context

        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

        Question: {query}

        Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"



In [ ]:
llm = GroqLLM()

Initialized Groq LLM with model: llama-3.1-8b-instant


### Enhanced RAG pipeline feature

In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(
        query,
        top_k=top_k,
        score_threshold=min_score
    )

    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])

    sources = [{
        'source': doc['metadata'].get(
            'source_file',
            doc['metadata'].get('source', 'unknown')
        ),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]

    confidence = max(
        [doc['similarity_score'] for doc in results]
    )

    # Generate answer
    response = llm.generate_response(query, context)

    output = {
        'answer': response,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context

    return output

In [ ]:
# Example usage:
answer = rag_advanced("What Is OPT?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=False)
print("Answer:", answer)


Retrieving documents for query: 'What Is OPT?'
Top K: 3, Score threshold: 0.1
Generating embeddinggs for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.14it/s]

Generated embedding with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}


In [ ]:
answer = rag_advanced("What Is the Purpose of Form I-765?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=False)
print("Answer:", answer)


Retrieving documents for query: 'What Is EAD?'
Top K: 3, Score threshold: 0.1
Generating embeddinggs for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 68.88it/s]

Generated embedding with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: {'answer': 'Based on the provided context, an Employment Authorization Document (EAD) is a card (Form I-766, or any successor document) issued as evidence that the holder is authorized to work in the United States.', 'sources': [{'source': 'i-765instr.pdf', 'page': 0, 'score': 0.24069952964782715, 'preview': 'employment with a specific employer under 8 CFR 274a.12(b), do not use Form I-765.\nDefinition\nEmployment Authorization Document (EAD):  The EAD is the card (Form I-766, or any successor document) issued as \nevidence that the holder is authorized to work in the United States.\nInitial EAD:  An EAD iss...'}, {'source': 'i-765instr.pdf', 'page': 0, 'score': 0.11061441898345947, 'preview': 'Renewal EAD:  An EAD issued to an eligible applicant after the expiration of a previous EAD issued under the same \ncategory.\nReplacement EAD:  An EAD issued to an eligible applicant when the previously issued EAD was lost, stolen, damaged, \nor contains errors, such as a misspelled nam